# NOTE:

This Jupyter Notebook is tested at personal workspaces/environment.
Tested using this specification:
  - CPU: AMD Ryzen 5 5600
  - Memory: 16GB DDR4-3200Mhz
  - GPU: NVIDIA RTX 3060 12G
  - Storage: NVME M.2 Up to 6,000MB/s
  - Operating System: Ubuntu-24.04 (WSL2)

Your steps or results may vary.

## Library Import

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from keras import Model, Sequential
from keras.layers import Dense, Input
from keras.optimizers import Adam
from keras.metrics import Mean
from keras.callbacks import ModelCheckpoint, EarlyStopping


## Dataset

In [ ]:
df_train = pd.read_csv('train.csv')

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(df_train)

## Model

### Encoder

In [ ]:
latent_dim = 2
input_shape = X_train.shape[1]

input_layer = Input(shape=(input_shape,))
encoder = Dense(64, activation='relu')(input_layer)
encoder = Dense(48, activation='relu')(encoder)
encoder = Dense(16, activation='relu')(encoder)

latent_encoding = Dense(latent_dim, activation='relu')(encoder)


### Decoder

In [ ]:
decoder = Dense(16, activation='relu')(latent_encoding)
decoder = Dense(48, activation='relu')(decoder)
decoder = Dense(64, activation='relu')(decoder)

output_layer = Dense(input_shape, activation='relu')(decoder)

### Autoencoder Model

In [ ]:
autoencoder_model = Model(input_layer, output_layer)
autoencoder_model.summary()

### Training

In [ ]:
model_name = 'autoencoder_model.weights.h5'
checkpoint = ModelCheckpoint(model_name,
                             monitor='val_loss',
                             mode='min',
                             save_best_only=True,
                             save_weights_only=True,
                             verbose=1)

earlystop = EarlyStopping(monitor='val_loss',
                          min_delta=0,
                          patience=5,
                          verbose=1,
                          restore_best_weights=True)

callback = [checkpoint, earlystop]

In [ ]:
autoencoder_model.compile(optimizer=Adam(learning_rate=0.001), loss='mae')
training = autoencoder_model.fit(X_train, X_train,
                                 epochs=50, batch_size=512,
                                 callbacks=callback)

# References
- [Intro to Autoencoders - Tensorflow](https://www.tensorflow.org/tutorials/generative/autoencoder)
- [Deep Learning: Part III Chapter 14 - Goodfellow et al.](https://www.deeplearningbook.org/)